In [1]:
import sys
sys.path.append("..")

In [2]:
import os
import tqdm
import torch
import pickle
import kaleido
import warnings
import numpy as np
import pandas as pd
import torch.optim as optim
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go

from src.data import *
from src.utils import *
from src.model import *
from src.recourse import *

warnings.filterwarnings('ignore')

In [3]:
def append_result(d, objective, loss, cost, m1_validity, wc_validity, m1_expectation, wc_expectation):
    d['cost'].append(cost)
    d['m1_validity'].append(m1_validity)
    d['wc_validity'].append(wc_validity)
    d['m1_probability'].append(m1_expectation)
    d['wc_probability'].append(wc_expectation)
    d['loss'].append(loss)
    d['J'].append(objective) 
    
def get_result(d, algorithm, seed, alpha, lamb, theta_0, theta_r):
    result = {
        'algorithm': algorithm, 
        'seed': seed,
        'alpha': alpha,
        'lambda': lamb,
        'theta_0': theta_0,
        'theta_r': theta_r
        }
    
    for key in d.keys():
        result[key] = np.mean(d[key])
    return result

In [4]:
def get_model_adv_pga_linf(X_0, X_r, cfr, alpha, lamb, pga_max_iter: int = 100):
    X_0 = torch.tensor(X_0).float()
    X_r = torch.tensor(X_r).float()
    weights_min = [cfr.model[0].weight.detach().clone()-alpha, cfr.model[2].weight.detach().clone()-alpha, cfr.model[4].weight.detach().clone()-alpha, cfr.model[6].weight.detach().clone()-alpha]
    weights_max = [cfr.model[0].weight.detach().clone()+alpha, cfr.model[2].weight.detach().clone()+alpha, cfr.model[4].weight.detach().clone()+alpha, cfr.model[6].weight.detach().clone()+alpha]
    bias_min = [cfr.model[0].bias.detach().clone()-alpha, cfr.model[2].bias.detach().clone()-alpha, cfr.model[4].bias.detach().clone()-alpha, cfr.model[6].bias.detach().clone()-alpha]
    bias_max = [cfr.model[0].bias.detach().clone()+alpha, cfr.model[2].bias.detach().clone()+alpha, cfr.model[4].bias.detach().clone()+alpha, cfr.model[6].bias.detach().clone()+alpha]
    
    cfr_adv = deepcopy(cfr)

    for module in cfr_adv.model.children():
        if isinstance(module, torch.nn.Linear):
            torch.nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                torch.nn.init.constant_(module.bias, 0.01)

    # optimizer = optim.Adam(cfr_adv.model.parameters(), maximize=True)
    optimizer = optim.SGD(cfr_adv.model.parameters(), lr=1, maximize=True)
    loss_fn = torch.nn.BCELoss(reduction='mean')

    loss = torch.tensor(1.)
    loss_diff = 1
    i = 0
    # for epoch in range(pga_max_iter):
    while loss_diff > 1e-4:
        if i == pga_max_iter:
            break

        prev_loss = loss.clone().detach()
        optimizer.zero_grad()
        
        f_x = cfr_adv.model(X_r)
        y_target = torch.ones(f_x.shape).to(torch.float32)
        bce_loss = loss_fn(f_x, y_target)
        loss = bce_loss
        
        loss.backward()
        optimizer.step()
        
        loss_diff = torch.dist(prev_loss, loss, 1)
        i += 1
        
        with torch.no_grad():
            # clamp model parameters to -alpha, alpha range
            cfr_adv.model[0].weight.clamp_(weights_min[0], weights_max[0])
            cfr_adv.model[2].weight.clamp_(weights_min[1], weights_max[1])
            cfr_adv.model[4].weight.clamp_(weights_min[2], weights_max[2])
            cfr_adv.model[6].weight.clamp_(weights_min[3], weights_max[3])
            cfr_adv.model[0].bias.clamp_(bias_min[0], bias_max[0])
            cfr_adv.model[2].bias.clamp_(bias_min[1], bias_max[1])
            cfr_adv.model[4].bias.clamp_(bias_min[2], bias_max[2])
            cfr_adv.model[6].bias.clamp_(bias_min[3], bias_max[3])

    return cfr_adv

In [5]:
def project_l1_ball(x: torch.Tensor, eps: float) -> torch.Tensor:
    """
    Project x onto the L1-ball {z : ||z||_1 <= eps}.
    Works for any shape. Preserves device/dtype.
    """
    # Flatten
    orig_shape = x.shape
    v = x.detach().reshape(-1)

    # Already feasible
    if torch.linalg.norm(v, ord=1) <= eps:
        return x

    # Sort |v| descending
    u, _ = torch.sort(v.abs(), descending=True)
    sv = torch.cumsum(u, dim=0)

    j = torch.arange(1, u.numel() + 1, device=v.device, dtype=v.dtype)
    # Find rho = max { j : u_j > (sv_j - eps)/j }
    cond = u * j > (sv - eps)
    rho = torch.nonzero(cond, as_tuple=False).max()
    theta = (sv[rho] - eps) / (rho.to(v.dtype) + 1)

    # Soft-threshold with theta and reshape back
    w = torch.sign(v) * torch.clamp(v.abs() - theta, min=0)
    return w.view(orig_shape)

In [6]:
def get_model_adv_pga_l1(X_0, X_r, cfr, alpha, lamb, pga_max_iter: int = 100):
    X_0 = torch.tensor(X_0).float()
    X_r = torch.tensor(X_r).float()
    
    cfr_adv = deepcopy(cfr)

    for module in cfr_adv.model.children():
        if isinstance(module, torch.nn.Linear):
            torch.nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                torch.nn.init.constant_(module.bias, 0.01)

    optimizer = optim.SGD(cfr_adv.model.parameters(), lr=0.1, maximize=True)
    loss_fn = torch.nn.BCELoss(reduction='mean')

    loss = torch.tensor(1.)
    loss_diff = 1
    i = 0
    while loss_diff > 1e-4:
    # for epoch in range(pga_max_iter):
        if i == pga_max_iter:
            break
        
        prev_loss = loss.clone().detach()
        optimizer.zero_grad()
        
        f_x = cfr_adv.model(X_r)
        y_target = torch.ones(f_x.shape).to(torch.float32)
        bce_loss = loss_fn(f_x, y_target)
        loss = bce_loss
        
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            for param_adv, param in zip(cfr_adv.model.parameters(), cfr.model.parameters()):
                delta = param_adv - param
                delta_proj = project_l1_ball(delta, alpha)
                param_adv.copy_(param + delta_proj)
        
        loss_diff = torch.dist(prev_loss, loss, 1)
        i += 1
    return cfr_adv

In [7]:
def evaluate_performance_one_nn(X_0, X_r, alpha, lamb, seed, alg, theta_adv_method='Linf', dataset=None):
    results = {'cost': [], 'm1_validity': [], 'wc_validity': [], 'm1_probability': [], 'wc_probability': [], 'loss': [], 'J': []}

    cfr = NN(X_0.shape[1])
    cfr.model.load_state_dict(torch.load(f"../results/recourse_model/{dataset}_{seed}.pth"))

    if alpha != 0:
        if theta_adv_method=='L-1':
            cfr_adv = get_model_adv_pga_l1(X_0, X_r, cfr, alpha, lamb, 99)
        else:
            cfr_adv = get_model_adv_pga_linf(X_0, X_r, cfr, alpha, lamb, 99)
    else:
        cfr_adv = deepcopy(cfr)        
    n = len(X_r)

    for i in tqdm.trange(n, desc=f'[{alg.capitalize()}] [ seed={seed} ] [ α={alpha} ] [ λ={lamb} ]', colour='#0091ff'):
        x_0 = X_0[i]
        x_r = X_r[i]
        J = RecourseCost(x_0, lamb)
        
        bce_loss_opt, cost_opt, rob_opt = J.eval_nonlinear(x_r.reshape((1,len(x_r))), cfr_adv.model, True)
        m1_validity_opt = cfr.predict(x_r.reshape(1,-1))[0]
        m1_expectation_opt = cfr.predict_proba(x_r.reshape(1,-1))[0,1]
        
        wc_validity_opt = cfr_adv.predict(x_r.reshape(1,-1))[0]
        wc_expectation_opt = cfr_adv.predict_proba(x_r.reshape(1,-1))[0,1]
        
        append_result(results, rob_opt, bce_loss_opt, cost_opt, m1_validity_opt, wc_validity_opt, m1_expectation_opt, wc_expectation_opt)
        
    return get_result(results, alg, seed, alpha, lamb, None, None)

In [8]:
def evaluate_performance_many_nn(X_0, X_r, alpha, lamb, seed, alg, theta_adv_method='Linf', dataset=None):
    results = {'cost': [], 'm1_validity': [], 'wc_validity': [], 'm1_probability': [], 'wc_probability': [], 'loss': [], 'J': []}

    cfr = NN(X_0.shape[1])
    cfr.model.load_state_dict(torch.load(f"../results/recourse_model/{dataset}_{seed}.pth"))       
    n = len(X_r)

    for i in tqdm.trange(n, desc=f'[{alg.capitalize()}] [ seed={seed} ] [ α={alpha} ] [ λ={lamb} ]', colour='#0091ff'):
        x_0 = X_0[i]
        x_r = X_r[i]
        if theta_adv_method=='L-1':
            cfr_adv = get_model_adv_pga_l1(x_0, x_r, cfr, alpha, lamb, 99)
        else:
            cfr_adv = get_model_adv_pga_linf(x_0, x_r, cfr, alpha, lamb, 99)
        J = RecourseCost(x_0, lamb)
        
        bce_loss_opt, cost_opt, rob_opt = J.eval_nonlinear(x_r.reshape((1,len(x_r))), cfr_adv.model, True)
        m1_validity_opt = cfr.predict(x_r.reshape(1,-1))[0]
        m1_expectation_opt = cfr.predict_proba(x_r.reshape(1,-1))[0,1]
        
        wc_validity_opt = cfr_adv.predict(x_r.reshape(1,-1))[0]
        wc_expectation_opt = cfr_adv.predict_proba(x_r.reshape(1,-1))[0,1]
        
        append_result(results, rob_opt, bce_loss_opt, cost_opt, m1_validity_opt, wc_validity_opt, m1_expectation_opt, wc_expectation_opt)
        
    return get_result(results, alg, seed, alpha, lamb, None, None)

In [9]:
def get_theta_adv_linf(X_0, X_r, theta_0, alpha, lamb):
    theta_adv = deepcopy(theta_0)
    
    for i in range(theta_0.shape[0]):
        theta_r_min = deepcopy(theta_0)
        theta_r_max = deepcopy(theta_0)
        
        theta_r_min[i] -= alpha
        theta_r_max[i] += alpha
        weights_r_min, bias_r_min = theta_r_min[:-1], theta_r_min[[-1]]
        weights_r_max, bias_r_max = theta_r_max[:-1], theta_r_max[[-1]]
        J_min, J_max = [], []
        for xi in range(len(X_r)):
            x_0 = X_0[xi]
            x_r = X_r[xi]
            J = RecourseCost(x_0, lamb)
            j_min = J.eval(x_r, weights_r_min, bias_r_min)
            j_max = J.eval(x_r, weights_r_max, bias_r_max)
            J_min.append(j_min)
            J_max.append(j_max)
        
        
        if np.mean(J_min) > np.mean(J_max):
            theta_adv[i] -= alpha
        else:
            theta_adv[i] += alpha

    return theta_adv

In [10]:
def get_theta_adv_l1(X_0, X_r, theta_0, alpha, lamb):
    thetas = generateThetas(theta_0, alpha)
    if alpha == 0:
        return thetas[0].copy()
    
    Js = np.empty((X_0.shape[0], thetas.shape[0]))
    for i in range(X_0.shape[0]):
         J = RecourseCost(X_0[i], lamb)
         for j, theta in enumerate(thetas):  
            Js[i, j] = J.eval(X_r[i], theta[:-1], np.array([theta[-1]]))
    
    Js_sum = Js.sum(axis=0) 
    Js_sum_maxI = np.argmax(Js_sum)
    theta_adv = thetas[Js_sum_maxI]

    return theta_adv

def generateThetas(theta0 : np.ndarray, alpha):
        # theta0 has bias
        thetas = theta0.copy()
        if alpha == 0:
            return np.array([thetas])
        
        thetas = np.repeat(thetas.reshape(1, theta0.size), (theta0.size * 2) - 1, axis=0)
        thetas_i = 0

        for i in range(theta0.size):
            if i == theta0.size - 1:
                thetas[thetas_i][i] -= alpha
                thetas_i += 1
                break

            thetas[thetas_i][i] += alpha
            thetas_i += 1
            thetas[thetas_i][i] -= alpha
            thetas_i += 1

        return thetas

In [11]:
def evaluate_performance(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method='L-inf'):
    results = {'cost': [], 'm1_validity': [], 'wc_validity': [], 'm1_probability': [], 'wc_probability': [], 'loss': [], 'J': []}
    weights_0, bias_0 = theta_0[:-1], theta_0[[-1]]

    if alpha != 0:
        if theta_adv_method=='L-1':
            theta_adv = get_theta_adv_l1(X_0, X_r, theta_0, alpha, lamb)
        else:
            theta_adv = get_theta_adv_linf(X_0, X_r, theta_0, alpha, lamb)
    else:
        theta_adv = theta_0.copy()
    
    weights_adv, bias_adv = theta_adv[:-1], theta_adv[[-1]]
        
    n = len(X_r)
    Y_0 = np.hstack((np.ones((n//2,)), np.zeros((n - n//2,))))
    clf = LR()
    clf.train(X_0, Y_0)
    clf.model.coef_ = weights_0.reshape(1,-1)
    clf.model.intercept_ = bias_0
    
    clf_adv = deepcopy(clf)
    clf_adv.model.coef_ = weights_adv.reshape(1,-1)
    clf_adv.model.intercept_ = bias_adv

    for i in tqdm.trange(n, desc=f'[{algorithm.capitalize()}] [ seed={seed} ] [ α={alpha} ] [ λ={lamb} ]', colour='#0091ff'):
        x_0 = X_0[i]
        x_r = X_r[i]
        J = RecourseCost(x_0, lamb)
        
        bce_loss, cost, price = J.eval(x_r, weights_adv, bias_adv, True)
        m1_validity = clf.predict(x_r.reshape(1,-1))[0]
        m1_probability = clf.predict_proba(x_r.reshape(1,-1))[0,1]
        
        wc_validity = clf_adv.predict(x_r.reshape(1,-1))[0]
        wc_probability = clf_adv.predict_proba(x_r.reshape(1,-1))[0,1]
        
        append_result(results, price, bce_loss, cost, m1_validity, wc_validity, m1_probability, wc_probability)
        
    return get_result(results, algorithm, seed, alpha, lamb, theta_0, theta_adv)

In [12]:
def calThetaAdv_l1(xP: np.ndarray, theta0: np.ndarray, alpha):
    # xP has bias
    thetaP = theta0.copy()
    i = np.argmax(np.abs(xP))
    thetaP[i] -= (alpha * np.sign(xP[i]))

    return thetaP

def calThetaAdv_linf(xP: np.ndarray, weights: np.ndarray, bias, alpha):
    # xP does not have bias
    weights_adv = weights - (alpha * np.sign(xP))

    for i in range(len(xP)):
        if np.sign(xP[i]) == 0:
            weights_adv[i] = weights_adv[i] - (alpha * np.sign(weights_adv[i]))
    bias_adv = bias - alpha

    return np.concat((weights_adv, bias_adv))

def calTheta(xP: np.array, weights: np.array, bias: np.array, alpha: float, methods: str):
    if methods == "L-inf":
        thetaP = calThetaAdv_linf(xP, weights, bias, alpha)
    else:
        thetaP = calThetaAdv_l1(np.hstack((xP, np.array([1]))), np.hstack((weights, bias)), alpha)

    return thetaP[:-1], np.array([thetaP[-1]])

In [13]:
def evaluate_performance_each(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method='L-inf'):
    results = {'cost': [], 'm1_validity': [], 'wc_validity': [], 'm1_probability': [], 'wc_probability': [], 'loss': [], 'J': []}
    
    n = len(X_r)
    Y_0 = np.hstack((np.ones((n//2,)), np.zeros((n - n//2,))))
    
    clf = LR()
    clf.train(X_0, Y_0)
    clf_adv = deepcopy(clf)


    for i in tqdm.trange(n, desc=f'[{algorithm.capitalize()}] [ seed={seed} ] [ α={alpha} ] [ λ={lamb} ]', colour='#0091ff'):
        x_0 = X_0[i]
        x_r = X_r[i]
        t_0 = theta_0[i]
        w_0, b_0 = t_0[:-1], t_0[[-1]]
        if alpha != 0:
            w_0_adv, b_0_adv = calTheta(x_r, w_0, b_0, alpha, theta_adv_method)
        else:
            w_0_adv, b_0_adv = w_0.copy(), b_0.copy()

        clf.model.coef_ = w_0.reshape(1,-1)
        clf.model.intercept_ = b_0
        clf_adv.model.coef_ = w_0_adv.reshape(1,-1)
        clf_adv.model.intercept_ = b_0_adv

        J = RecourseCost(x_0, lamb)
        bce_loss, cost, price = J.eval(x_r, w_0_adv, b_0_adv, True)

        m1_validity = clf.predict(x_r.reshape(1,-1))[0]
        m1_probability = clf.predict_proba(x_r.reshape(1,-1))[0,1]
        
        wc_validity = clf_adv.predict(x_r.reshape(1,-1))[0]
        wc_probability = clf_adv.predict_proba(x_r.reshape(1,-1))[0,1]
        
        append_result(results, price, bce_loss, cost, m1_validity, wc_validity, m1_probability, wc_probability)
        
    return get_result(results, algorithm, seed, alpha, lamb, None, None)

In [14]:
def evaluate_performance_shifted(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, dataset):
    results = {'cost': [], 'm1_validity': [], 'wc_validity': [], 'm1_probability': [], 'wc_probability': [], 'loss': [], 'J': []}
    
    n = len(X_r)
    Y_0 = np.hstack((np.ones((n//2,)), np.zeros((n - n//2,))))
    
    clf = LR()
    clf.train(X_0, Y_0)
    
    with open(f"../results/recourse_model/lr_shift_{dataset}_{seed}.pkl", "rb") as f:
        clf_adv = pickle.load(f)
    w_adv, b_adv = clf_adv.model.coef_[0], clf_adv.model.intercept_
    t_r = np.hstack((w_adv, b_adv))
    
    for i in tqdm.trange(n, desc=f'[{algorithm.capitalize()}] [ seed={seed} ] [ λ={lamb} ]', colour='#0091ff'):
        x_0 = X_0[i]
        x_r = X_r[i]
        t_0 = theta_0[i]
        w_0, b_0 = t_0[:-1], t_0[[-1]]
        clf.model.coef_ = w_0.reshape(1,-1)
        clf.model.intercept_ = b_0
        
        J = RecourseCost(x_0, lamb)
        bce_loss, cost, price = J.eval(x_r, w_adv, b_adv, True)

        m1_validity = clf.predict(x_r.reshape(1,-1))[0]
        m1_probability = clf.predict_proba(x_r.reshape(1,-1))[0,1]
        
        wc_validity = clf_adv.predict(x_r.reshape(1,-1))[0]
        wc_probability = clf_adv.predict_proba(x_r.reshape(1,-1))[0,1]
        
        append_result(results, price, bce_loss, cost, m1_validity, wc_validity, m1_probability, wc_probability)
        
    return get_result(results, algorithm, seed, alpha, lamb, t_0, t_r)

In [15]:
def evaluate_performance_nn_shifted(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, dataset):
    results = {'cost': [], 'm1_validity': [], 'wc_validity': [], 'm1_probability': [], 'wc_probability': [], 'loss': [], 'J': []}

    cfr = NN(X_0.shape[1])
    cfr.model.load_state_dict(torch.load(f"../results/recourse_model/{dataset}_{seed}.pth"))       
    with open(f"../results/recourse_model/nn_shift_{dataset}_{seed}.pkl", "rb") as f:
        cfr_adv = pickle.load(f)
    n = len(X_r)

    for i in tqdm.trange(n, desc=f'[{algorithm.capitalize()}] [ seed={seed} ] [ λ={lamb} ]', colour='#0091ff'):
        x_0 = X_0[i]
        x_r = X_r[i]
        J = RecourseCost(x_0, lamb)
        
        bce_loss_opt, cost_opt, rob_opt = J.eval_nonlinear(x_r.reshape((1,len(x_r))), cfr_adv.model, True)
        m1_validity_opt = cfr.predict(x_r.reshape(1,-1))[0]
        m1_expectation_opt = cfr.predict_proba(x_r.reshape(1,-1))[0,1]
        
        wc_validity_opt = cfr_adv.predict(x_r.reshape(1,-1))[0]
        wc_expectation_opt = cfr_adv.predict_proba(x_r.reshape(1,-1))[0,1]
        
        append_result(results, rob_opt, bce_loss_opt, cost_opt, m1_validity_opt, wc_validity_opt, m1_expectation_opt, wc_expectation_opt)
        
    return get_result(results, algorithm, seed, alpha, lamb, None, None)

In [16]:
def runCostValidityTradeoff (results: dict, params: dict):
    dataset = params["data"]
    base_model = params["base_model"]
    for algorithm in params['algorithms']:
        for seed in params['seeds']:
            for v_alpha in params['alpha']:
                for v_lamb in params["lambda"][base_model][dataset][algorithm][v_alpha]:
                    data = pd.read_pickle(f"../results/recourse/{params['base_model']}_{params['data']}_{algorithm}_{v_lamb}_{v_alpha}_{seed}.pkl")
                    alpha = data["alpha"].unique().item()
                    lamb = data["lambda"].unique().item()
                    X_0 = np.stack(data["x_0"])
                    X_r = np.stack(data["x_r"])
                    theta_0 = np.stack(data["theta_0"])
                    
                    # if (params['include_mask'] and algorithm != "L1PSD"):
                    #     data_l1psd = pd.read_pickle(f"../results/recourse/{params['base_model']}_{params['data']}_L1PSD_0.1_0.1_{seed}.pkl")
                    #     # data_l1psd = pd.read_pickle(f"../results/recourse/{params['base_model']}_{params['data']}_L1PSD_0.01_{v_alpha}_{seed}.pkl")
                    #     mask_i = data_l1psd["i"].to_numpy()
                    #     X_0 = X_0[mask_i]
                    #     X_r = X_r[mask_i]
                    #     theta_0 = theta_0[mask_i]
                    # else:
                    #     data_l1psd = pd.read_pickle(f"../results/recourse/{params['base_model']}_{params['data']}_L1PSD_0.1_0.1_{seed}.pkl")
                    #     mask_i = data_l1psd["i"].to_numpy()
                    #     if params['data'] == "sba" and params['base_model'] == "lr" and v_alpha == 0.5:
                    #         data = data[data['i'].isin(mask_i)]    
                    #     elif params['data'] == "sba" and params['base_model'] == "nn" and v_alpha == 0.5:
                    #         data = data[data['i'].isin(mask_i)]

                    #     X_0 = np.stack(data["x_0"])
                    #     X_r = np.stack(data["x_r"])
                    #     theta_0 = np.stack(data["theta_0"])
                            
                    match params['adv_method']:
                        case "ONE":
                            if params['base_model'] == 'lr':
                                res = evaluate_performance(X_0, X_r, theta_0[0], alpha, lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
                            elif params['base_model'] == 'nn':
                                res =evaluate_performance_one_nn(X_0, X_r, alpha, lamb, seed, algorithm, "L-1" if "L1" in algorithm else "L-inf", dataset=params['data'])
                        case "MANY":
                            # res = evaluate_performance_each(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
                            if params['base_model'] == 'lr':
                                res = evaluate_performance_each(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
                            elif params['base_model'] == 'nn':
                                res =evaluate_performance_many_nn(X_0, X_r, alpha, lamb, seed, algorithm, "L-1" if "L1" in algorithm else "L-inf", dataset=params['data'])
                        case "LARGESTALPHA":
                            res = evaluate_performance(X_0, X_r, theta_0[0], max(params['alphas']), lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
                        case "SMALLESTALPHA":
                            res = evaluate_performance(X_0, X_r, theta_0[0], min(params['alphas']), lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
                        case "THETA0":
                            if params['base_model'] == 'lr':
                                res = evaluate_performance_each(X_0, X_r, theta_0, 0, lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
                            elif params['base_model'] == 'nn':
                                res =evaluate_performance_one_nn(X_0, X_r, 0, lamb, seed, algorithm, "L-1" if "L1" in algorithm else "L-inf", dataset=params['data'])      
                        case "SHIFTED":
                            if params['base_model'] == 'lr':
                                res = evaluate_performance_shifted(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, dataset=params['data'])
                            elif params['base_model'] == 'nn':
                                res = evaluate_performance_nn_shifted(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, dataset=params['data'])
                        case "_":
                            print(f"{params['adv_method']} does not exist!")

                    results['algorithm'].append(algorithm)
                    results['seed'].append(seed)
                    results['alpha'].append(alpha)
                    results['lambda'].append(lamb)
                    results['Cost'].append(res['cost'])
                    results['Current Validity'].append(res['m1_probability'])
                    results['Worst Case Validity'].append(res['wc_probability'])
                    results['BCE Loss'].append(res['loss'])
                    results['J'].append(res['J'])
    
    df_results = pd.DataFrame(results)
    return df_results

In [42]:
params = {}
# 'lr', 'nn'
params['base_model'] = 'nn'
# 'synthetic', 'german', 'sba', 'income'
params['data'] = 'income'
params['seeds'] = range(2)
# 'Alg1', 'L1PSD', 'ROARLInf', 'ROARL1'
params['algorithms'] = ['Alg1', 'L1PSD']
# 'ONE', 'MANY', 'LARGESTALPHA', 'SMALLESTALPHA', 'THETA0', "SHIFTED"
params['adv_method'] = 'MANY'
params['include_base_model'] = False
params['include_mask'] = True
params["alpha"] = [0.1]
params["lambda"] = {
    "lr": {
        "german": {
            "Alg1": {
                0.1: np.hstack((np.arange(0.01, 0.105, 0.01),np.arange(0.2,0.55, 0.1), np.array([0.008, 0.004, 0.001]))).round(5),
                0.5:  np.hstack((np.arange(0.01, 0.105, 0.01),np.arange(0.2,0.55, 0.1), np.array([0.008, 0.004]))).round(5)
            },
            "L1PSD": {
                0.1: [0.5,0.3,0.1,0.04,0.01,0.004,0.001],
                0.5: [0.3,0.04, 0.004, 0.1, 0.01]
            },
            "ROARLInf": {
                0.1: np.hstack((np.array([0.008, 0.004, 0.001]),np.arange(0.01, 0.105, 0.01),np.arange(0.2,0.55, 0.1))).round(5),
                0.5: np.hstack((np.array([0.004, 0.008]),np.arange(0.01, 0.105, 0.01),np.arange(0.2,0.55, 0.1))).round(5)
            },
            "ROARL1": {
                0.1: np.hstack((np.array([0.008, 0.004, 0.001]),np.arange(0.01, 0.105, 0.01),np.arange(0.2,0.55, 0.1))).round(5),
                0.5: np.hstack((np.array([0.004, 0.008]),np.arange(0.01, 0.105, 0.01),np.arange(0.2,0.55, 0.1))).round(5)
            }
        },
        "sba": {
            "Alg1": {
                0.1: [0.01, 0.1, 0.7, 1.4, 2.1],
                0.5:  [0.01, 0.1, 0.7, 1.4, 2.1]
            },
            "L1PSD": {
                0.1: [0.01, 0.1, 0.7, 1.4, 2.1],
                0.5: [0.01, 0.1, 0.7, 1.4, 2.1]
            },
            "ROARLInf": {
                0.1: [0.01, 0.08, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0, 1.1, 1.4, 1.5, 1.6, 1.8, 2, 2.1],
                0.5: np.hstack((np.arange(0.01,0.095,0.01), np.array([0.01, 0.08, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0, 1.1, 1.4, 1.5, 1.6, 1.8, 2, 2.1]))).round(5)
            },
            "ROARL1": {
                0.1: [0.01, 0.08, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0, 1.1, 1.4, 1.5, 1.6, 1.8, 2, 2.1],
                0.5: np.hstack((np.arange(0.01,0.095,0.01), np.array([0.01, 0.08, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0, 1.1, 1.4, 1.5, 1.6, 1.8, 2, 2.1]))).round(5)
            }
        },
        "income": {
            "Alg1": {
                0.1: [0.01, 0.1, 0.5, 0.7, 1.0, 1.5],
                0.5: [0.01, 0.1, 0.5, 0.7, 1.0, 1.5]
            },
            "L1PSD": {
                # 0.1: [0.01, 0.1, 0.5],
                0.1: [0.01, 0.1],
                0.5: [0.01, 0.1]
            },
            "ROARLInf": {
                0.1: [0.01, 0.1, 0.5],
                0.5: [0.01, 0.1, 0.5]
            },
            "ROARL1": {
                0.1: [0.01, 0.1, 0.5],
                0.5: [0.01, 0.1, 0.5]
            }
        }
    },
    "nn": {
        "german": {
            "Alg1": {
                0.1: np.hstack((np.arange(0.01, 0.105, 0.01),np.arange(0.2,1.05, 0.1), np.array([2.0, 3.0]))).round(5),
                0.5:  np.hstack((np.arange(0.01, 0.105, 0.01),np.arange(0.2,1.05, 0.1), np.array([2.0, 3.0]))).round(5)
            },
            "L1PSD": {
                0.1: [3.0, 0.7, 0.3, 0.1, 0.05, 0.01],
                0.5: [3.0, 0.7, 0.3, 0.1, 0.05, 0.01]
            },
            "ROARLInf": {
                0.1: np.hstack((np.arange(0.01, 0.105, 0.01),np.arange(0.2,1.05, 0.1), np.array([2.0, 3.0]))).round(6),
                0.5: np.hstack((np.arange(0.01, 0.105, 0.01),np.arange(0.2,1.05, 0.1), np.array([2.0, 3.0]))).round(7)
            },
            "ROARL1": {
                0.1: np.hstack((np.arange(0.01, 0.105, 0.01),np.arange(0.2,1.05, 0.1), np.array([2.0, 3.0]))).round(6),
                0.5: np.hstack((np.arange(0.01, 0.105, 0.01),np.arange(0.2,1.05, 0.1), np.array([2.0, 3.0]))).round(7)
            }
        },
        "sba": {
            "Alg1": {
                0.1: [0.01, 0.1, 0.7, 1.4, 2.1, 2.8, 3.5],
                0.5:  [0.01, 0.1, 0.7, 1.4, 2.1, 2.8, 3.5]
            },
            "L1PSD": {
                0.1: [0.01, 0.1, 0.7, 1.4, 2.1, 2.8, 3.5],
                0.5: [0.01, 0.1, 0.7, 2.1, 3.5]
            },
            "ROARLInf": {
                0.1: [0.01, 0.1, 0.3, 0.5, 0.7, 1.4, 1.7, 1.9, 2.1, 2.8, 3.5],
                0.5: np.hstack((np.arange(0.01, 0.105, 0.01),np.arange(0.2,1.05, 0.1), np.array([1.4, 2.1, 2.8, 3.5]))).round(7)
            },
            "ROARL1": {
                0.1: [0.01, 0.1, 0.3, 0.5, 0.7, 1.4, 1.7, 1.9, 2.1, 2.8, 3.5],
                0.5: np.hstack((np.arange(0.01, 0.105, 0.01),np.arange(0.2,1.05, 0.1), np.array([1.4, 2.1, 2.8, 3.5]))).round(7)
            }
        },
        "income": {
            "Alg1": {
                0.1: [0.0001, 0.001, 0.01, 0.1, 0.5],
                0.5: [0.0001, 0.001, 0.01, 0.1, 0.5]
            },
            "L1PSD": {
                0.1: [0.01, 0.1, 0.5],
                0.5: [0.01, 0.1, 0.5]
            },
            "ROARLInf": {
                0.1: [0.01, 0.1, 0.5],
                0.5: [0.01, 0.1, 0.5]
            },
            "ROARL1": {
                0.1: [0.01, 0.1, 0.5],
                0.5: [0.01, 0.1, 0.5]
            }
        }
    }
}



results = {
    'algorithm': [],
    'seed': [],
    'alpha': [],
    'lambda': [],
    'Cost': [],
    'Current Validity': [],
    'Worst Case Validity': [],
    'BCE Loss': [],
    'J': []
}


# f_name = f"../results/cost_validity_satml/model_{params['base_model']}-data_{params['data']}-alpha_{params["alpha"][0]}-adv_method_{params["adv_method"].lower()}.pkl"
# if os.path.exists(f_name):
#     df_results = pd.read_pickle(f_name)
# else:
#     df_results = runCostValidityTradeoff(results, params)
#     df_results.to_pickle(f_name)

df_results = runCostValidityTradeoff(results, params)
# df_results.to_pickle(f_name)

[Alg1] [ seed=0 ] [ α=0.1 ] [ λ=0.0001 ]:   0%|          | 0/16 [00:00<?, ?it/s]

[L1psd] [ seed=1 ] [ α=0.1 ] [ λ=0.5 ]: 100%|██████████| 16/16 [00:02<00:00,  7.60it/s]


In [43]:
df_results_avg = df_results.groupby(['algorithm', 'lambda', 'alpha'], as_index=False).mean()
df_results_avg

,algorithm,lambda,alpha,seed,Cost,Current Validity,Worst Case Validity,BCE Loss,J
0,Alg1,0.0001,0.1,0.5,8.901784,0.457651,1.732458e-18,76.791433,76.792323
1,Alg1,0.0010,0.1,0.5,6.870778,0.493460,4.559199e-18,79.406457,79.413327
2,Alg1,0.0100,0.1,0.5,4.737531,0.521908,1.239926e-18,86.224432,86.271807
3,Alg1,0.1000,0.1,0.5,2.398987,0.543082,2.324912e-18,86.759338,86.999237
4,Alg1,0.5000,0.1,0.5,1.821478,0.517589,1.338718e-18,90.384233,91.294972
5,L1PSD,0.0100,0.1,0.5,4.180497,0.527071,3.435644e-01,1.908656,1.950461
6,L1PSD,0.1000,0.1,0.5,2.160659,0.535062,3.686661e-01,1.148888,1.364954
7,L1PSD,0.5000,0.1,0.5,1.608484,0.499286,3.505290e-01,1.197376,2.001618


In [44]:
algorithm_map = {"Alg1" : "Alg2 (Linf)",
                "L1PSD" : "Alg (L1)",
                "ROARLInf" : "ROAR (Linf)",
                "ROARL1" : "ROAR (L1)"}
algorithm_latex_map = {"Alg10.1" : r"$\text{Alg}2\ (L^\infty)\ (\alpha=0.1)$",
                       "L1PSD0.1" : r"$\text{Alg}1\ (L^1)\ (\alpha=0.1)$",
                       "ROARLInf0.1" : r"$\text{ROAR}\ (L^\infty)\ (\alpha=0.1)$",
                       "ROARL10.1" : r"$\text{ROAR}\ (L^1)\ (\alpha=0.1)$",
                       "Alg10.5" : r"$\text{Alg}2\ (L^\infty)\ (\alpha=0.5)$",
                       "L1PSD0.5" : r"$\text{Alg}1\ (L^1)\ (\alpha=0.5)$",
                       "ROARLInf0.5" : r"$\text{ROAR}\ (L^\infty)\ (\alpha=0.5)$",
                       "ROARL10.5" : r"$\text{ROAR}\ (L^1)\ (\alpha=0.5)$"}

df_results_avg["algorithm"] = df_results_avg["algorithm"].replace(algorithm_map)

# params["algorithms"] = ["L1PSD", "Alg1", "ROARL1", "ROARLInf"]

In [45]:
data_map = {'synthetic': 'Synthetic', 'sba': 'Small Business Administration', 'german': 'German', 'income': 'ACS Income'}
model_map = {'lr': 'Logistic Regression', 'nn': 'Neural Network'}

'#636EFA',
'#EF553B',
'#00CC96',
'#AB63FA',
"#8EF1F3",
"#F6B08C",
"#B5FFBE",
"#D7BBF4"

custom_colors = {"Alg1" : '#636EFA', "L1PSD" : '#EF553B',"ROARLInf" : '#00CC96', "ROARL1" : '#AB63FA'}

In [46]:
def plotFigures(df_results_avg, params, should_plot_frontier=True):
    font_family = 'Times New Roman'
    font_color = 'black'
    font_size = 20
    width, height = 720, 540
    fig = go.Figure()

    for i, alg in enumerate(params["algorithms"]):
        df_alg = df_results_avg.copy()
        df_alg = df_alg[(df_alg['algorithm']==algorithm_map[alg])].sort_values(['Cost'], ascending=True).reset_index(drop=True)
        if should_plot_frontier:
            x, y, mask = find_pareto(df_alg['Cost'], df_alg['Worst Case Validity'], return_index=True)
        else:
            x, y, mask = df_alg['Cost'], df_alg['Worst Case Validity'], range(df_alg.shape[0])
        df_alg = pd.DataFrame({'Algorithm': [f"{alg}" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'alpha': df_alg['alpha'][mask], 'lambda': df_alg['lambda'][mask]})

        alpha = df_alg['alpha'].unique()[0]
        alg_alpha = alg+str(alpha)

        fig.add_trace(go.Scatter(
            x = df_alg['Cost'],
            y = df_alg['Worst Case Validity'],
            mode = 'lines+markers' if alg != 'wachter' else 'markers',
            name = algorithm_latex_map[alg_alpha],
            marker = dict(color=custom_colors[alg], size=5),
            showlegend=True,
            customdata=df_alg[['alpha', 'lambda']].to_numpy(),
            hovertemplate='Cost: %{x}<br>Validity: %{y}<br>alpha: %{customdata[0]}<br>lambda: %{customdata[1]}'
        ))

    fig.update_xaxes(
        title=dict(
            text='Implementation Cost',
            font=dict(
                family=font_family,
                color=font_color,
                size=font_size
            )
            ), 
        showline=True, 
        mirror=True,
        linecolor='black', 
        gridcolor='lightgrey', 
        zerolinewidth=1,
        zerolinecolor='lightgrey',
        )

    y_axis_title_text = {
        "MANY": "Instance-Wise Validity",
        "ONE": "Population-Wise Validity",
        "THETA0": "Current Validity",
        "SHIFTED": "Future Validity"
    }
    fig.update_yaxes(
        title=dict(
            text=y_axis_title_text[params["adv_method"]],
            font=dict(
                family=font_family,
                color=font_color,
                size=font_size
            ), 
            ), 
        showline=True, 
        mirror=True,
        linecolor='black', 
        gridcolor='lightgrey',
        zerolinewidth=1,
        zerolinecolor='lightgrey',
        )


    fig.update_layout(
        width=width,
        height=height,
        plot_bgcolor='white',
        paper_bgcolor='white',
        margin=dict(t=25,b=25,l=25,r=25),
        legend=dict(
            x=0.975, 
            y=0.025, 
            orientation='v',
            xanchor='right',
            font=dict(
                family=font_family,
                color=font_color,
                size=15
                ), 
            bgcolor='rgba(255, 255, 255, 0.7)',
            bordercolor='lightgrey',
            borderwidth=1,
            entrywidth=100.5,
            ),
        xaxis=dict(
            tickfont=dict(
                family=font_family,
                color=font_color,
                size=20,
            ),
            # alpha=0.1
            # range=[-0.5,21.5], # german_lr
            # range=[-0.2,10.5], # german_nn
            # range=[-0.2,5.2], # sba_lr
            # range=[-0.2,4.5], # sba_nn
            # alpha=0.5
            # range = [-0.2, 7] # sba_lr

            range = [-0.2, 9]
            
        ),
        yaxis=dict(
            tickfont=dict(
                family=font_family,
                color=font_color,
                size=20
            ),
            range=[-0.03,1.03],
        )
    )

    return fig

In [47]:
should_plot_frontier = False
fig = plotFigures(df_results_avg, params, should_plot_frontier)

print(f'{params["data"]}  |  {params["base_model"].upper()} | {params["adv_method"]}')
fig.show()

income  |  NN | MANY


In [92]:
# kaleido.get_chrome_sync()
validity_type = {
    "MANY": "instance_wise",
    "ONE": "population_wise",
    "THETA0": "current",
    "SHIFTED": "future"
}

if should_plot_frontier:
    figName = f"cost_validity-{validity_type[params["adv_method"]]}-{params['base_model']}-{params['data']}-alpha_{params["alpha"][0]}-frontiers"
else:
    figName = f"cost_validity-{validity_type[params["adv_method"]]}-{params['base_model']}-{params['data']}-alpha_{params["alpha"][0]}"
kaleido.write_fig_sync(fig, f"../fig/{figName}.pdf")